# LV with Coarsening Aware Loss on Synthetic AML dataset built with AMLGentex

## Import

In [1]:
import os
import sys
import random
import numpy as np

import warnings
warnings.filterwarnings("ignore")

# pygsp
from pygsp import graphs

# scipy
import scipy as sp

# torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# torch geometric
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

# sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

# datasets
from datasets import load_dataset

# plot
import matplotlib.pyplot as plt

# utils
from utils.split_dataset import *
from utils.visualization import *
from utils.coarsening import apply_Loukas_coarsening, create_pygsp_graph
from utils.aml_dataset import load_aml_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(1)
np.random.seed(1)

Using device: cuda


`FEATURES` N x D matrix:
- bankName (int)
- phoneChanges (int)
- daysInBank (int)

`TRANSACTIONS` S x M_s x D matrix:
- from (int)
- to (int)
- amount (float32)
- type (int)
- isSAR (int)
- alertID (int)
- modelType (int)


node mapping: `f(n) = n + 2`:
- Source node (Integration): `-2` --> 0
- Sink node (Placement): `-1` --> 1

Type mapping:
- INITALBALANCE: `0`
- CASH: `1`
- TRANSFER: `2`

## General functions

### Get coarsened data from Coarsening matrix C

In [2]:
def get_coarsened_edges_features_labels(C: list, Gc: graphs.Graph, features, labels, edges, weights, priority_label=1):
    """
    Coarsen edges and features based on the coarsening matrix C;
    the feature matrix (N x D) is made by summing up features from nodes that are coarsened into the same supernode.
    priority_label is 1 because it is the label for SAR transactions.

    Output:
      - edges_idx_coarsened: coarsened edges index (num_edges, 2)
      - features_coarsened: coarsened features (num_nodes_coarsened, num_features)
      - labels_coarsened: coarsened labels (num_nodes_coarsened,)
      - weights_coarsened: coarsened weights (num_edges_coarsened,)
    """

    idx_map = {} # map fine node -> coarse node 
    for supernode, node in zip(*C.nonzero()):
        idx_map[node] = supernode
        
    # features coarsened
    features_coarsened = torch.tensor(C @ features.cpu(), dtype=torch.float32)

    # edges coarsened
    edges_idx_coarsened = np.array(Gc.get_edge_list()[:2]).T

    # labels coarsened
    num_classes = 2
    all_labels = np.zeros((C.shape[0], num_classes), dtype=np.float32)
    for supernode, node in zip(*C.nonzero()):
        all_labels[supernode, labels[node]] += 1

    labels_coarsened = torch.zeros(C.shape[0], dtype=torch.long, device=device)
    for supernode in range(C.shape[0]):
        mx = np.argmax(all_labels[supernode])
        priority_count = all_labels[supernode, priority_label]
        if priority_count >= mx:
            labels_coarsened[supernode] = priority_label
        else: 
            labels_coarsened[supernode] = mx
    
    # weights coarsened
    weights_coarsened = []
    for i in range(len(edges)):
        src, dst = edges[i][0], edges[i][1]
        if idx_map[src] != idx_map[dst]:
            weights_coarsened.append(weights[i])
            
    weights_coarsened = torch.tensor(weights_coarsened, dtype=torch.float32, device=device)

    return edges_idx_coarsened, features_coarsened, labels_coarsened, weights_coarsened

## GNN

### model and training function

In [3]:
class GCN(nn.Module):
    def __init__(self, nfeat, nhid, nclass, dropout=.5):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(nfeat, nhid)
        self.conv2 = GCNConv(nhid, nclass)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.conv1(x, edge_index, edge_weight))
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv2(x, edge_index, edge_weight)
        return F.log_softmax(x, dim=1)
    
    def get_embeddings(self, x, edge_index, edge_weight=None):
        x = x.to(next(self.parameters()).device)
        edge_index = edge_index.to(x.device)
        if edge_weight is not None:
            edge_weight = edge_weight.to(x.device)
        x = F.relu(self.conv1(x, edge_index, edge_weight))
        return x

class CoarseningAwareLoss(nn.Module):
    def __init__(self, coarse_weight: float = 1.0):
        """
        Args:
          coarse_weight: weight for the coarsening loss term.
        """
        super().__init__()
        self.coarse_weight = coarse_weight
        # self.class_loss = nn.NLLLoss()
        self.class_loss = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 100.0], device=device))  # assuming class 1 is the priority class (SAR transactions)


    def forward(self,
            output: torch.Tensor,
            embeddings: torch.Tensor,
            labels: torch.Tensor,
            coarsening_matrix: sp.sparse.csr_matrix,
            train_idx: torch.Tensor):
        """
        output: [N, C] log-probabilities (log_softmax).
        embeddings: [N, D] raw features from model.get_embeddings().
        labels: [N] ground-truth class labels.
        coarsening_matrix: [Nc, N] coarsening matrix.
        train_idx: indices of coarsened nodes used for classification loss.
        """
        device = output.device
        N = embeddings.shape[0]

        # 1. Classification loss
        loss_cls = self.class_loss(output[train_idx], labels[train_idx])

        # 2. Embedding normalization
        embeddings_norm = F.normalize(embeddings, p=2, dim=1)

        with torch.no_grad():
            supernodes = torch.zeros(N, dtype=torch.long, device=device)
            for i, j in zip(*coarsening_matrix.nonzero()):
                supernodes[j] = i


        loss_coarse = torch.tensor(0.0, device=device)
        count = 0

        # nodes mapped in the same supernode
        # for supernode in range(coarsening_matrix.shape[0]):
        #     for row in zip(coarsening_matrix.getrow(supernode)):
        #         for col1 in row[0].nonzero()[1]:
        #             for col2 in row[0].nonzero()[1]:
        #                 if col1 != col2:
        #                     emb1 = embeddings_norm[col1]
        #                     emb2 = embeddings_norm[col2]
        #                     sim = F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).squeeze()
        #                     loss_coarse += (1 - sim) / 2.0


        n_sample = 2000
        for _ in range(n_sample):
            i, j = random.sample(range(N), 2)
            emb_i = embeddings_norm[i]
            emb_j = embeddings_norm[j]
            sim = F.cosine_similarity(emb_i.unsqueeze(0), emb_j.unsqueeze(0)).squeeze()

            # common
            if supernodes[i] == supernodes[j]:
                loss_coarse += 1 - sim
            else:
                loss_coarse += sim
            count += 1
            
            # node mapped in different supernodes
            # if supernodes[i] != supernodes[j]:
            #     loss_coarse += sim
            #     count += 1

        loss_coarse = loss_coarse / count if count > 0 else torch.tensor(0.0, device=device)
        return loss_cls + self.coarse_weight * loss_coarse

In [4]:
def create_pyg_data(features, edges_idx, labels, weights=None) -> Data:
    edge_index = torch.as_tensor(edges_idx.T, dtype=torch.long)

    if weights is not None:
        edge_attr = torch.as_tensor(weights, dtype=torch.float)
    else:
        edge_attr = None

    edge_index, edge_attr = to_undirected(edge_index, edge_attr)

    return Data(x=features, edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor(labels))


def train_gnn_1_epoch(model: nn.Module, optimizer: optim.Optimizer, criterion: nn.Module, data: Data, embeddings, coarsening_matrix, train_idx: list, val_idx: list):
    """
    Output:
        - train_loss
        - val_loss
        - val_accuracy
    """
    
    data = data.to(next(model.parameters()).device)

    model.train()
    optimizer.zero_grad()
    
    output = model(data.x, data.edge_index, data.edge_attr)
    # embeddings = model.get_embeddings(data.x, data.edge_index)

    # train
    loss = criterion(output, embeddings, data.y, coarsening_matrix, train_idx)
    loss.backward()
    optimizer.step()
    
    # validate
    model.eval()
    with torch.no_grad():
        output = model(data.x, data.edge_index, data.edge_attr)
        # embeddings = model.get_embeddings(data.x, data.edge_index)

        loss_val = criterion(output, embeddings, data.y, coarsening_matrix, val_idx)
        pred_val = output[val_idx].max(1)[1]
        acc_val = accuracy_score(data.y[val_idx].cpu().numpy(), pred_val.cpu().numpy())

    
    return loss.item(), loss_val.item(), acc_val

def evaluate_model(model: nn.Module, data: Data, test_idx, log_info=True):
    """Evaluate the model on test set."""
    model.eval()
    data = data.to(next(model.parameters()).device)
    
    with torch.no_grad():
        output = model(data.x, data.edge_index, data.edge_attr)
        pred_test = output[test_idx].max(1)[1]
        acc_test = accuracy_score(data.y[test_idx].cpu().numpy(), pred_test.cpu().numpy())
        
        if log_info:
            print(f'\nTest Accuracy: {acc_test:.4f}')
            print('\nClassification Report:')
            print(classification_report(
                data.y[test_idx].cpu().numpy(), pred_test.cpu().numpy())
            )

    return acc_test, pred_test

## `Coarsen - GNN and Loss`

**workflow:**
```latex
1. Train GNN on full graph to initialize weights
2. Iterate:
    1. generate embeddings of all nodes from GNN(A, X)
    2. coarsen the graph using embeddings as node features
    3. Train GNN on coarsened nodes
        - loss is coarsening-aware
```

In [ ]:
# AML_DATASET = load_aml_dataset()
AML_DATASET = load_aml_dataset(step_start=0, steps_amount=20, extend_features=False)
EDGES = AML_DATASET['edges']
WEIGHTS = AML_DATASET['weights']
FEATURES = AML_DATASET['features']
LABELS = AML_DATASET['ground_truth']
NUM_NODES = AML_DATASET['num_nodes']

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f136103fcd0>>
Traceback (most recent call last):
  File "/home/white/miniconda3/envs/dml/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [ ]:
def train_GNN_coarsening_aware_loss(epochs: int, lr=0.01, wd=5e-4, method='variation_neighborhoods', ratio=0.8, similarity_threshold=0.65):
    # model and criterion
    nfeat = FEATURES.shape[1]
    nhid = 128
    nclass = 2
    dropout = 0.1
    model = GCN(nfeat=nfeat, nhid=nhid, nclass=nclass, dropout=dropout).to(device)
    criterion = CoarseningAwareLoss()

    # train data
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    data = create_pyg_data(FEATURES, EDGES, LABELS, WEIGHTS)

    x, ycrs, yfine, ylosst, ylossv, valacc = [], [], [], [], [], []
    for epoch in range(epochs):

        print(f"Iteration {epoch + 1:02d}/{epochs}")

        # get embeddings from the GNN
        embeddings = model.get_embeddings(data.x, data.edge_index, data.edge_attr)

        # coarsen the graph using embeddings as features
        G = create_pygsp_graph(edges=EDGES, num_nodes=data.num_nodes, weights=WEIGHTS)
        
        C, Gc, Call, Gall = apply_Loukas_coarsening(
            G, X=embeddings, method=method, ratio=ratio, K=50, similarity_threshold=similarity_threshold, log_info=True
        )

        # train the GNN on the coarsened graph
        edges_idx_c, feat_c, labels_c, weights_c = get_coarsened_edges_features_labels(
            C, Gc, FEATURES, LABELS, EDGES, WEIGHTS, priority_label=0
        )

        data_c = create_pyg_data(feat_c, edges_idx_c, labels_c, weights_c)
        
        train_idx_c, val_idx_c, test_idx_c = create_train_val_test_split(data_c.num_nodes)
        train_idx_c = torch.LongTensor(train_idx_c).to(device)
        val_idx_c = torch.LongTensor(val_idx_c).to(device)
        test_idx_c = torch.LongTensor(test_idx_c).to(device)


        train_loss, validation_loss, validation_accuracy = train_gnn_1_epoch(
            model, optimizer, criterion, data_c, embeddings, C, train_idx_c, val_idx_c
        )

        print(f"Train Loss: {train_loss:.4f}, Validation Loss: {validation_loss:.4f}, Validation Accuracy: {validation_accuracy:.4f}")

        # evaluate the model on the coarsened graph
        acc_test_c, pred_test_c = evaluate_model(model, data_c, test_idx_c, log_info=False)

        # evaluate on original (fine) graph
        pred_fine = torch.tensor([-1 for _ in range(NUM_NODES)], dtype=torch.int64, device=device)
        tot = 0
        for idx in range(len(test_idx_c)):
            test_supernode = test_idx_c[idx]
            for fine_node in C.getrow(test_supernode).nonzero()[1]:
                pred_fine[fine_node] = pred_test_c[idx]
                tot += 1

        sm = torch.sum(pred_fine == LABELS)
        accuracy_fine = sm / tot

        illicit_misclassified = torch.sum((pred_fine != LABELS) & (LABELS == 1))
        tot_illicit = torch.sum(LABELS == 1)
        good_misclassified = torch.sum((pred_fine != LABELS) & (LABELS == 0))
        tot_good = torch.sum(LABELS == 0)

        print(f"Test - coarse accuracy: {acc_test_c:.4f}, fine accuracy: {accuracy_fine:.4f}")
        print(f"illicit misclassified: {illicit_misclassified}/{tot_illicit}, good_misclassified:{good_misclassified}/{tot_good}\n")

        # ploting data
        x.append(epoch + 1)
        ycrs.append(acc_test_c)
        yfine.append(accuracy_fine)
        ylosst.append(train_loss)
        ylossv.append(validation_loss)
        valacc.append(validation_accuracy)

    np.save(f'saves/data_AML_gnn_CoarseningAwareLoss_threshold_{similarity_threshold*100:.0f}_{method}_ratio_{ratio*100:.0f}.npy', {
        'x': x,
        'ycrs': ycrs,
        'yfine': yfine,
        'ylosst': ylosst,
        'ylossv': ylossv,
        'valacc': valacc,
        'description': f'Data obtained using: {ratio=}, {method=}, {similarity_threshold=}, CoarseningAwareLoss()'
    })

    torch.save(model.state_dict(), f'../models/data_AML_gnn_CoarseningAwareLoss_threshold_{similarity_threshold*100:.0f}_{method}_ratio_{ratio*100:.0f}.pt')


## main

In [ ]:
from itertools import product

epochs = 50
methods = ['variation_edges']
# methods = ['variation_neighborhoods']
# ratios = [.5, .8, .9, .95]
# thresholds = [0.50, 0.65, 0.75, 0.85]
ratios = [.8]
thresholds = [0.75]

for method, ratio, threshold in product(methods, ratios, thresholds):
    train_GNN_coarsening_aware_loss(epochs=epochs, method=method, ratio=ratio, similarity_threshold=threshold)

# per iteration:
#   coarsening ~50 sec
#   training   ~15 sec

Iteration 01/50
Coarsening: variation_edges (99998 n, 329474 e) -> (10000 n, 59003 e); 
Train Loss: 3.4756, Validation Loss: 25.7228, Validation Accuracy: 0.7750
Test - coarse accuracy: 0.7710, fine accuracy: 0.3824
illicit misclassified: 61600/61600, good_misclassified:30684/38398

Iteration 02/50
Coarsening: variation_edges (99998 n, 329474 e) -> (10000 n, 59034 e); 
Train Loss: 21.2066, Validation Loss: 2.2110, Validation Accuracy: 0.2350
Test - coarse accuracy: 0.2150, fine accuracy: 0.6147
illicit misclassified: 48861/61600, good_misclassified:38398/38398

Iteration 03/50
Coarsening: variation_edges (99998 n, 329474 e) -> (10000 n, 58994 e); 
Train Loss: 2.4191, Validation Loss: 3.8126, Validation Accuracy: 0.2560
Test - coarse accuracy: 0.2245, fine accuracy: 0.6169
illicit misclassified: 48684/61600, good_misclassified:38398/38398

Iteration 04/50


KeyboardInterrupt: 